### Step 1 — Point at the same catalog/schema as Notebook 02

In [0]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
documents_table = f"{catalog_name}.{schema_name}.documents"

print(f"Documents table: {documents_table}")

### Step 2 — Load the labeled documents

Same table Notebook 02 wrote. `category` here is the ground truth we hand-assigned -- `ai_classify` never sees this column, only `content`.

In [0]:
documents_df = spark.table(documents_table)
documents_df.createOrReplaceTempView("documents_v")
display(documents_df.select("doc_id", "title", "category"))

### Step 3 — Classify every document with `ai_classify`, using the bare category names

`ai_classify(content, labels)` takes the text to classify and an array of candidate labels, and returns one of them. We pass exactly the five category names Notebook 02 used, so a "correct" answer here means an exact string match against `actual_category`.

In [0]:
%sql
SELECT
  doc_id,
  title,
  category AS actual_category,
  ai_classify(
    content,
    ARRAY('Product', 'Operations', 'Compliance', 'Customer Service', 'Technical')
  ) AS predicted_category
FROM documents_v
ORDER BY doc_id;

### Step 4 — Capture the result in Python and score it against ground truth

Same query, kept as a DataFrame this time so we can compute accuracy and pull out the specific rows `ai_classify` got wrong -- those mismatches are more informative than the accuracy number itself.

In [0]:
classified_df = spark.sql(
    """
    SELECT
      doc_id,
      title,
      category AS actual_category,
      ai_classify(
        content,
        ARRAY('Product', 'Operations', 'Compliance', 'Customer Service', 'Technical')
      ) AS predicted_category
    FROM documents_v
    """
)

total = classified_df.count()
correct = classified_df.filter("actual_category = predicted_category").count()
print(f"Accuracy on {total} labeled documents: {correct}/{total} ({100 * correct / total:.1f}%)")

print("\nMismatches (worth reading, not just counting):")
display(classified_df.filter("actual_category != predicted_category"))

In [0]:
from pyspark.sql.functions import col

documents_df.filter((col('doc_id').isin(4, 14))).display()

### Step 5 — Re-run with descriptive labels instead of bare names

Same documents, same query shape -- only the label *wording* changes, from a bare category name to a one-line description of what belongs in it. If any mismatch from Step 4 disappears here, that's a concrete demonstration of label wording changing model behavior, not the underlying document changing.

In [0]:
import json

descriptive_labels = {
    "Product": "describes a bank account, deposit, or credit product and its terms",
    "Operations": "describes an internal operational or branch procedure, such as cash or wire handling",
    "Compliance": "describes a regulatory, KYC, AML, or data privacy policy",
    "Customer Service": "describes a customer-facing support process, such as disputes or card replacement",
    "Technical": "describes an API, authentication, or engineering reference document",
}

labels_json = json.dumps(descriptive_labels)
labels_sql_string = labels_json.replace("'", "''")

classified_v2_df = spark.sql(
    f"""
    SELECT
        doc_id,
        title,
        category AS actual_category,
        ai_classify(
            content,
            '{labels_sql_string}',
            MAP(
                'version', '2.1', 
                'enableConfidenceScores', 'true',
                'enableRationales', 'true'
            )
        ) AS predicted_category_v2
    FROM documents_v
"""
)

classified_v2_df = classified_v2_df.selectExpr(
            "doc_id",
            "title",
            "actual_category",
            "REPLACE(predicted_category_v2:response[0]:value, '\"', '') AS predicted_category_v2",
            "REPLACE(predicted_category_v2:response[0]:confidence_score, '\"', '') AS confidence_score",
            "REPLACE(predicted_category_v2:response[0]:rationale, '\"', '') AS rationale"
        )

comparison_df = classified_df.join(classified_v2_df, on=["doc_id", "title", "actual_category"])
display(
    comparison_df.select(
        "doc_id", "title", "actual_category", "predicted_category", "predicted_category_v2", "confidence_score", "rationale"
    )
)

### Step 6 — Classify an unlabeled, genuinely ambiguous document

The Fraud Escalation Playbook from Notebook 04 was never assigned a Notebook-02-style category. Its text is reconstructed here directly (not re-parsed via `ai_parse_document`), so this step doesn't depend on that notebook's live, workspace-specific output schema -- only on the PDF's known content.

In [0]:
fraud_escalation_text = (
    "Fraud Escalation Playbook\n\n"
    "This playbook defines how Aurora Trust Bank staff escalate suspected fraud cases. "
    "Any transaction flagged by fraud monitoring must be reviewed within one business hour. "
    "Confirmed fraud cases are escalated to the Fraud Operations team, who freeze the affected "
    "account and notify the customer through the contact center. "
    "Escalation Steps: (1) Analyst reviews the flagged transaction. (2) Analyst confirms or "
    "dismisses the fraud indicator. (3) Confirmed cases are routed to Fraud Operations. "
    "(4) Fraud Operations freezes the account and opens a case file. "
    "Escalation Tiers: Tier 1 handles amounts under 1,000 units; Tier 2 handles 1,000-10,000 "
    "units and requires a supervisor sign-off; Tier 3 handles amounts above 10,000 units and "
    "requires notifying the Compliance department."
)

fraud_df = spark.createDataFrame([("fraud_escalation_playbook", fraud_escalation_text)], ["title", "content"])
fraud_df.createOrReplaceTempView("fraud_doc_v")

display(
    spark.sql(
        """
        SELECT
          title,
          ai_classify(
            content,
            ARRAY('Product', 'Operations', 'Compliance', 'Customer Service', 'Technical')
          ) AS predicted_category
        FROM fraud_doc_v
        """
    )
)

### Step 7 — Validate the function's own contract, not just the answer

Separately from whether the classification is *correct*, confirm the returned value is actually a member of the label array you supplied. `ai_classify` is documented to guarantee this by construction, but this notebook hasn't been run against a live workspace -- checking it once, explicitly, is cheaper than assuming it and finding out otherwise three notebooks from now.

In [0]:
allowed_categories = ["Product", "Operations", "Compliance", "Customer Service", "Technical"]

invalid_predictions = classified_df.filter(~classified_df.predicted_category.isin(allowed_categories))
invalid_count = invalid_predictions.count()

if invalid_count == 0:
    print(f"All {classified_df.count()} predictions are members of the supplied label set -- contract holds.")
else:
    print(f"{invalid_count} prediction(s) fell outside the supplied label set:")
    display(invalid_predictions)

### Step 8 - Exercising with PDF file

Read the PDF in `binary format` -> parse the document using `ai_parse_document` -> concatenate some values -> classify using `ai_classify`

In [0]:
pdf_volume_path = "/Volumes/main/genai_lab/synthetic_data/documents/16_fraud_escalation_playbook.pdf"
pdf_binary_df = spark.read.format("binaryFile").load(pdf_volume_path)
pdf_binary_df.createOrReplaceTempView("fraud_pdf_raw")

In [0]:
from pyspark.sql.functions import col, posexplode, expr, concat_ws, collect_set

# Step 1: Parse the document using ai_parse_document (SQL function)
parsed_df = spark.sql("""
    SELECT 
        path, 
        ai_parse_document(
            content, 
            MAP('version', '2.0')
        ) AS parsed
    FROM fraud_pdf_raw
""")

# Step 2: Explode the elements array with position, then extract VARIANT fields
# posexplode returns (pos, col) — equivalent to variant_explode on a VARIANT array
# try_cast converts the VARIANT array to ARRAY<VARIANT> so posexplode can iterate
# element:field accesses VARIANT sub-fields (same : syntax as SQL)
elements_df = (
    parsed_df
    .select(
        col("path"),
        posexplode(expr("try_cast(parsed:document:elements AS ARRAY<VARIANT>)")).alias("pos", "element")
    )
    .select(
        col("path"),
        col("pos").alias("element_position"),
        expr("element:confidence").alias("confidence"),
        expr("element:content").alias("content"),
        expr("element:description").alias("description"),
        expr("element:id").alias("id"),
        expr("element:type").alias("type"),
    )
)

display(elements_df)

In [0]:
from pyspark.sql import functions as F

pdf_content_concat = (
    elements_df
    .withColumn("id_int", F.col("id").cast("int"))
    .withColumn("content_string", F.col("content").cast("string"))
    # .filter(F.col("id_int").isin(0, 1, 3, 5))
    .filter(F.col("id_int").isin(3, 5))
    .groupBy("path")
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    F.col("id_int"),
                    F.col("content_string")
                )
            )
        ).alias("ordered_elements")
    )
    .select(
        col("path").alias("pdf_path"),
        F.concat_ws(
            "\n\n",
            F.col("ordered_elements.content_string")
        ).alias("content")
    )
)

display(pdf_content_concat)

In [0]:
classified_df = (
    pdf_content_concat
    .withColumn(
        "predicted_category",
        F.expr("""
            ai_classify(
                content,
                ARRAY('Product', 'Operations', 'Compliance', 'Customer Service', 'Technical', 'Fraud')
            )
        """)
    )
    .select(
        "pdf_path",
        "content",
        "predicted_category"
    )
)

classified_df.display()